# Machine Learning project in SoSe 2025 at HTW Saar
## Idea
The goal of this project is predicting the genre(s) of a game/bundle through its given description(s)

## Dataset
For our project we use a Steam Dataset provided on moodle, since it has all information we plan on using.
The Dataset has been cut to only 2000 data points to be runnable on weaker devices.

In [ ]:
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(transform_output="pandas")

dataset = pd.read_csv("./games_march2025_cleaned_2k.csv",sep=",")
print(dataset.head(1))

   appid              name release_date  required_age  price  dlc_count  \
0    730  Counter-Strike 2   2012-08-21             0    0.0          1   

                                detailed_description  \
0  For over two decades, Counter-Strike has offer...   

                                      about_the_game  \
0  For over two decades, Counter-Strike has offer...   

                                   short_description reviews  ...  \
0  For over two decades, Counter-Strike has offer...     NaN  ...   

  average_playtime_2weeks median_playtime_forever median_playtime_2weeks  \
0                     879                    5174                    350   

  discount  peak_ccu                                               tags  \
0        0   1212356  {'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...   

   pct_pos_total  num_reviews_total pct_pos_recent  num_reviews_recent  
0             86            8632939             82               96473  

[1 rows x 47 columns]


## Preparation of the Dataset
### Removing Uniques
We would remove the following features from the Training-Set as they can/could uniquely identify a datapoint, but we don't as they will be removed in the next step anyway
- AppId
- Name of the Game
- Realease Date
- Reviews
- Header Image
- Website
- Support URL
- Support Email
- MetaCritic URL
- Developer
- Publisher
- Screenshots
- Movies
- Estimated Owners

In [ ]:
#dataset.drop(['appid', 'name', 'release_date', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'metacritic_url', 'notes', 'developers', 'publishers', 'screenshots', 'movies', 'estimated_owners'], axis=1, inplace=True)
#print(dataset.head())

## Hold onto necessary information
Our model should turn a textual description of a game into its genre. For that we need all the textual information a game has, as well as the genres of the game.
We use a ColumnTransformer to drop all unnecessary lines, merge all descriptions of a game into one big description and hold onto the genres

It is important to use ``verbose_feature_names_out=False`` so the feature names don't get changed

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

# desc, genres
column_transformer = ColumnTransformer([
        # merge all descriptions
        ('desc', FunctionTransformer(lambda X: X.fillna('').agg(' '.join, axis=1).to_frame(name="desc")),
            ['detailed_description', 'about_the_game', 'short_description']),
        ('pass', 'passthrough', ['genres']),
    ],
    verbose_feature_names_out=False
)
dataset = column_transformer.fit_transform(dataset)
print(dataset.head())

                                                desc  \
0  For over two decades, Counter-Strike has offer...   
1  LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...   
2  The most-played game on Steam. Every day, mill...   
3  When a young street hustler, a retired bank ro...   
4  Edition Comparison Ultimate Edition The Tom Cl...   

                                              genres  
0                         ['Action', 'Free To Play']  
1  ['Action', 'Adventure', 'Massively Multiplayer...  
2             ['Action', 'Strategy', 'Free To Play']  
3                            ['Action', 'Adventure']  
4                                         ['Action']  


### Adding missing Information
Some Games might not have any descriptions. For these we Input an Empty String
**TODO: check if dropna and fillna numeric_only is needed, as we dont have any numbers**

In [ ]:
# missing numeric values => mean
dataset.fillna(dataset.mean(numeric_only=True), inplace=True)
# missing strings => empty string?
dataset.fillna('', inplace=True)
# drop all lines with missing values
dataset.dropna(inplace=True)

## Transform Genres
The genre information currently is a string holding a python array of genres. While this is machine-readable, we need One-Hot-Encoding for our model to work.

#### Serializing the String-Array
The "ast" library can interpret python strings as python code, and as such will be used for serializing the genres.

In [ ]:
import ast

dataset['genres'] = dataset['genres'].map(lambda s: ast.literal_eval(s))
print(dataset['genres'].head())

0                               [Action, Free To Play]
1    [Action, Adventure, Massively Multiplayer, Fre...
2                     [Action, Strategy, Free To Play]
3                                  [Action, Adventure]
4                                             [Action]
Name: genres, dtype: object


#### One-Hot-Encoding an Python-Array
The sklearn ``OneHotEncoder()`` is only able to work with an 1D Array of different classes, such as ``['Politics', 'Sport', 'Culture']``. Every datapoint can only have one concurrent classification.
Steam allows an app/bundle to have multiple genres. As such, our dataset has an 2D Array of different classes, which sklearn's ``MultiLabelBinarizer()`` does support.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_genres = MultiLabelBinarizer()
genres_encoded = mlb_genres.fit_transform(dataset.pop('genres'))
genres_df = pd.DataFrame(genres_encoded, columns=mlb_genres.classes_)
print(genres_df.head())

   Action  Adventure  Casual  Early Access  Free To Play  Gore  Indie  \
0       1          0       0             0             1     0      0   
1       1          1       0             0             1     0      0   
2       1          0       0             0             1     0      0   
3       1          1       0             0             0     0      0   
4       1          0       0             0             0     0      0   

   Massively Multiplayer  RPG  Racing  Simulation  Sports  Strategy  Violent  
0                      0    0       0           0       0         0        0  
1                      1    0       0           0       0         0        0  
2                      0    0       0           0       0         1        0  
3                      0    0       0           0       0         0        0  
4                      0    0       0           0       0         0        0  


With this, our target matrix is completed.

### Structurizing Text
If we want our Model to be able to use text as an input, we have to vectorize the text. TF-IDF (Inverse Document Frequency) is an easy way of transforming each word into a feature with a 0 to 1 value. **TODO: filter out stopwords**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(dataset['desc']) # matrix, not pandas df
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(tfidf_df.head())

    00  000  000km    000th  00am  00f  00i  00p  00v   01  ...  이터널  이터널리턴  \
0  0.0  0.0    0.0  0.00000   0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0    0.0   
1  0.0  0.0    0.0  0.00000   0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0    0.0   
2  0.0  0.0    0.0  0.14649   0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0    0.0   
3  0.0  0.0    0.0  0.00000   0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0    0.0   
4  0.0  0.0    0.0  0.00000   0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0    0.0   

   이현준  정대찬  중입니다   철권  토탈워  페르소나  한국어  한글을  
0  0.0  0.0   0.0  0.0  0.0   0.0  0.0  0.0  
1  0.0  0.0   0.0  0.0  0.0   0.0  0.0  0.0  
2  0.0  0.0   0.0  0.0  0.0   0.0  0.0  0.0  
3  0.0  0.0   0.0  0.0  0.0   0.0  0.0  0.0  
4  0.0  0.0   0.0  0.0  0.0   0.0  0.0  0.0  

[5 rows x 29351 columns]


With this our feature matrix is completed

In [ ]:
X = tfidf_df
y = genres_df

## The Model

####  Removing unpredicatble Datapoints
Some Datapoints don't have a genre assigned (all feature values in y are 0). The model we use can't handle such cases, thus they have to be removed.
We filter after all values that we can use with a mask, and apply that mask to our matrices.

In [ ]:
mask = y.sum(axis=1).map(lambda x: x > 0)
print((mask == False).sum()) # count of unpredictable datapoints

X_clean = X[mask]
y_clean = y[mask]

13


# Splitting up data
We have to split up our data into training and testing data.
Using random_state=0 guarantees reproducability.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, random_state=0)

# Model Selection
**TODO Deciding which model to use for this task**

As a game can have multiple genres, our Model(s) has to be capable of multi-label-classification. sklearn's ``MultiOutputClassifier`` can do this. As a backend for ``MultiOutputClassifier`` we use ``LogisticRegression``

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier

# n_jobs=1 since there seems to be some multithreading join issue in sklearn (or my pc is to bad)
multi_target_clf = MultiOutputClassifier(LogisticRegression(max_iter=1337, random_state=0), n_jobs=1)

multi_target_clf.fit(X_train, y_train)

y_pred = multi_target_clf.predict(X_test)

# Evaluation
**TODO Test the Model with the test data**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, zero_division=0.0))

              precision    recall  f1-score   support

           0       0.78      0.91      0.84       300
           1       0.78      0.62      0.69       216
           2       1.00      0.03      0.07        86
           3       0.00      0.00      0.00        46
           4       1.00      0.04      0.07        83
           5       0.00      0.00      0.00         0
           6       0.79      0.81      0.80       245
           7       0.00      0.00      0.00        42
           8       0.90      0.34      0.49       127
           9       0.00      0.00      0.00        12
          10       0.89      0.25      0.39       127
          11       0.00      0.00      0.00        14
          12       0.88      0.14      0.24       106
          13       0.00      0.00      0.00         0

   micro avg       0.79      0.50      0.61      1404
   macro avg       0.50      0.22      0.26      1404
weighted avg       0.77      0.50      0.53      1404
 samples avg       0.77   

# Optimization
**TODO optimize the model based on the test results**

# Validation
**TODO Predict actual values**

# Conclusion and outlook
**TODO Write a conclusion and outlook what can be done and where the issues were.**